# Brief 01 - "remaniement" complet - corrigé

In [2]:
import os
from pathlib import Path
import subprocess
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values 

> ⚠️ Il faut faire attention à bien modifier la variable `RELEVES_ROOT` en fonction de l'architecture de votre dossier / repository.

In [3]:
RELEVES_ROOT = "../releves"
# On ne sélectionne que les dossier grâce à l'usage de `.is_dir()` 
SITES = sorted(d.name for d in Path(RELEVES_ROOT).iterdir() if d.is_dir())
print(SITES)

['animalis', 'bitiba_fr', 'chronovet', 'clubvetshop', 'maxizoo', 'pharmacy4pets', 'univers_veto', 'vetoplus', 'vetostore', 'zooplus_fr']


In [4]:
# ⚠️ Il faut faire attention à bien modifier la variable DATABASE_URL, avec vos valeurs
DATABASE_URL = "dbname=vetprice" # "postgresql://<user>:<password>@<host>:<port>/<dbname>"
conn = psycopg2.connect(DATABASE_URL)
conn.autocommit = False
cur = conn.cursor()

## 1. Profilage et modélisation

### Organisation des données

Chaque site a été scrapé plusieurs fois : 
- une fois en avril
- deux en juillet

Un "run" (i.e. une exécution d'un scraper pour un site) produit un dossier horodaté `AAAA-MM-JJ_HHMM`, qui contient les produits collectés ce jour-là. Rien n'est écrasé d'un run à l'autre : 🔥🔥🔥 **c'est ce qui rend l'historisation possible**.

```
releves/
├── chronovet/
│   ├── 2026-04-01_1919/          run du 1er avril à 19h19
│   ├── 2026-07-12_1150/          run du 12 juillet à 11h50  (partiel car interrompu)
│   ├── 2026-07-12_1242/          run du 12 juillet à 12h42  (complet)
│   └── 2026-07-18_1334/          run du 18 juillet à 13h34
├── univers_veto/
│   ├── 2026-04-01_1847/
│   ├── 2026-04-01_1918/
│   ├── 2026-07-12_1148/
│   ├── 2026-07-12_1150/
│   ├── 2026-07-12_1242/
│   ├── 2026-07-18_1334/
│   └── 2026-07-18_1355/
├── animalis/    …
├── bitiba_fr/   …
├── clubvetshop/ …
├── maxizoo/     …
├── pharmacy4pets/ …
├── vetoplus/    …
├── vetostore/   …
└── zooplus_fr/  …
```

Un run contient : 

```
2026-07-18_1334/
├── products.jsonl    un produit par ligne, au format JSON (le champ scraped_at porte l'instant de collecte)
└── meta.json         pas toujours présent, métadonnées du run (products_scraped, http_requests, http_errors, completed, resumed)
```

**Pourquoi y a-t-il plusieurs dossiers le même jour ?**

- Un run peut être interrompu (site lent, coupure, mise en veille de l'ordinateur portable) puis relancé.
- Chaque relance (i.e. nouveau run) crée un nouveau dossier horodaté (i.e. le dossier du run).
- Par exemple, sur chronovet le 12 juillet, `_1150` est un run partiel et `_1242` un run complet du même jour.
- Le champ `HHMM` du nom permet de savoir lequel est le dernier


### Nombre de fichiers de relevés (i.e. un fichier par run)

In [5]:
RELEVES = [
    (site, run_dir / "products.jsonl")
    for site in SITES
    for run_dir in sorted((Path(RELEVES_ROOT) / site).iterdir())
    if (run_dir / "products.jsonl").exists()
]
print(len(RELEVES), "relevés (fichiers) au total")

41 relevés (fichiers) au total


### Profiler tous les fichiers

In [6]:
# Profilage sur TOUS les fichiers : chaque run de chaque site
def profiler_tous_les_fichiers():
    rows = []
    for site in SITES:
        for run_dir in (Path(RELEVES_ROOT) / site).iterdir():
            f = run_dir / "products.jsonl"
            # S'il n'y a pas de fichier pour ce dossier de run
            # on passe au `run_dir_ suivant dans la boucle (avec `continue`)
            if not f.exists():
                continue
            df = pd.read_json(f, lines=True, dtype=False)
            n = len(df)
            # # ean absent : soit colonne manquante, soit valeur vide (ou NaN, donc à bien vérifier)
            if "ean" in df.columns:
                ean = df["ean"]
                # Ci-dessous, ce n'est pas la façon la plus propre, 
                # mais cette écriture permet d'éviter une étape préliminaire de cleaning
                a_ean = ean.notna() & (ean.astype(str).str.strip() != "")
            else:
                ean, a_ean = pd.Series([None] * n, dtype=object), pd.Series([False] * n)
            rows.append({
                "site": site,
                "run": run_dir.name,
                "lignes": n,
                "sans_ean": int((~a_ean).sum()),
                "doublons_ean": int(ean[a_ean].duplicated().sum()),  # doublons internes au run
                "colonnes": df.shape[1],
            })
    return pd.DataFrame(rows)

In [7]:
stats = profiler_tous_les_fichiers()

# 💫 On ajoute le pourcentage de lignes sans ean par run
stats["sans_ean_%"] = (100 * stats["sans_ean"] / stats["lignes"]).round(1)
print(stats.to_string(index=False))

         site             run  lignes  sans_ean  doublons_ean  colonnes  sans_ean_%
     animalis 2026-07-12_1150     775         0           148        17         0.0
     animalis 2026-04-01_1919    2866         4           634        17         0.1
     animalis 2026-07-12_1242   10782         7          3434        17         0.1
     animalis 2026-07-18_1334   53561       820         31114        17         1.5
    bitiba_fr 2026-04-01_1921   21008         0          8062        20         0.0
    bitiba_fr 2026-07-12_1242   12763         0             0        12         0.0
    bitiba_fr 2026-07-18_1334   12659         0             0        12         0.0
    chronovet 2026-07-12_1150      90         0             0        19         0.0
    chronovet 2026-04-01_1919    1631         0             0        14         0.0
    chronovet 2026-07-12_1242    2436         5             0        19         0.2
    chronovet 2026-07-18_1334    2453         5             0        19     

### Agrégat par site (sur tous les runs)

In [8]:
agg = stats.groupby("site")[["lignes", "sans_ean", "doublons_ean"]].sum()
agg

,lignes,sans_ean,doublons_ean
site,,,
animalis,67984,831,35330
bitiba_fr,46430,0,8062
chronovet,6610,10,0
clubvetshop,18524,10581,8
maxizoo,18675,1017,1
pharmacy4pets,4120,4120,0
univers_veto,2146,135,0
vetoplus,7367,643,147
vetostore,10585,10585,0


### On calcule le pourcentage total de lignes sans ean

In [9]:
profils = {
    s: (int(r.lignes), r.sans_ean / r.lignes, int(r.doublons_ean)) 
    for s, r in agg.iterrows()
}


print(f"\nTotal : {int(stats['lignes'].sum())} lignes sur {len(stats)} fichiers, "
      f"{100 * agg['sans_ean'].sum() / agg['lignes'].sum():.1f}% sans ean global")


Total : 388784 lignes sur 41 fichiers, 7.2% sans ean global


### Il y a 7.2 % d'ean manquants -> il faudra une clef de repli
- ✅ on crée donc immédiatement une variable clé que l'on créera avant l'insertion

## 2. Chargement des relevés

## Toutes les données dans un seul dataframe

In [10]:
# On lit TOUTES les données (tous les runs de tous les sites)
frames = []
for site, chemin in RELEVES:
    d = pd.read_json(chemin, lines=True, dtype=False)
    d["date_releve"] = chemin.parent.name[:10]   # on sélectionne : AAAA-MM-JJ, la date du run
    frames.append(d)
df = pd.concat(frames, ignore_index=True)
print(len(df), "lignes chargées, tous sites et toutes dates confondus")

388784 lignes chargées, tous sites et toutes dates confondus


In [11]:
# Clé produit : l'ean s'il est présent et non vide, sinon l'url.
def cle_produit(row):
    ean = row.get("ean")
    if isinstance(ean, str) and ean.strip():
        return ean
    # Valeur par défaut (i.e. la solution de repli qu'on a choisi)
    return row["url"]

df["cle"] = df.apply(cle_produit, axis=1)
df[["site", "date_releve", "ean", "url", "cle"]].head()

,site,date_releve,ean,url,cle
0,animalis,2026-04-01,4015110034865,https://www.animalis.com/moser-tondeuse-1400-p...,4015110034865
1,animalis,2026-04-01,3182550702874,https://www.animalis.com/royal-canin-croquette...,3182550702874
2,animalis,2026-04-01,4014162613790,https://www.animalis.com/jbl-gant-de-nettoyage...,4014162613790
3,animalis,2026-04-01,4004218758865,https://www.animalis.com/tetra-traitement-gold...,4004218758865
4,animalis,2026-04-01,8010690030661,https://www.animalis.com/ferplast-roue-en-plas...,8010690030661


### Pourquoi `scraped_at` plutôt que `date_releve` ?

🔥 **Avant toute chose, ce n'est pas important de choisir l'un ou l'autre, les explications ci-dessous sont juste fournies à des fins de clarté**

- `date_releve` vient du nom du dossier (c'est le "slice" qui contient les 10 premiers caractères), il y en a un par dossier de run. `scraped_at` est difficile plus difficile à définir : pour faire simple : il y a **plus de `scraped_at` distincts que de `date_releve`**, surtout à cause de la reprise sur interruption : un run interrompu puis relancé réécrit dans le **même dossier** (donc même `date_releve`), mais les produits repris portent un **nouveau `scraped_at`**.

```
dossier  animalis/2026-07-18_1334/     ->  date_releve = "2026-07-18"  (10 premiers caractères)

products.jsonl :
   produit      1   scraped_at = 2026-07-18T13:34:00Z   ┐  run lancé à 13h34
   ...                                                   │  (629 lignes)
   produit    629   scraped_at = 2026-07-18T13:34:00Z   ┘
   ─── interruption (mise en veille), puis reprise ───
   produit    630   scraped_at = 2026-07-18T13:55:43Z   ┐  repris à 13h55,
   ...                                                   │  écrit dans le MÊME dossier
   produit  53561   scraped_at = 2026-07-18T13:55:43Z   ┘  (52 932 lignes)

   => 1 dossier, 1 date_releve, mais 2 scraped_at
```

On va donc choisir de dédoubloner sur `scraped_at` : deux observations à deux instants différents comptent comme deux versions, même si elles sont dans le même dossier.

In [12]:
# Table cible (on repart propre à chaque exécution).
cur.execute("DROP TABLE IF EXISTS produit_historise")
cur.execute("""
    CREATE TABLE produit_historise (
        site        text,
        cle         text,
        ean         text,
        url         text,
        name        text,
        brand       text,
        price       numeric,
        in_stock    boolean,
        valid_from  timestamptz,
        valid_to    timestamptz,
        is_current  boolean,
        PRIMARY KEY (site, cle, valid_from)
    )
""")
conn.commit()

# scraped_at partout : clé de dédoublonnage ET borne temporelle.
# Une version par (site, cle, scraped_at). Deux observations à deux instants
# différents sont deux versions, même dans le même dossier de run.
h = (df.sort_values("scraped_at")
       .drop_duplicates(["site", "cle", "scraped_at"], keep="last")
       .sort_values(["site", "cle", "scraped_at"])
       .copy())
h["valid_from"] = h["scraped_at"]
h["valid_to"] = h.groupby(["site", "cle"])["scraped_at"].shift(-1)  # scraped_at du relevé suivant du produit
h["is_current"] = h["valid_to"].isna()                             # dernière version connue

colonnes = ["site", "cle", "ean", "url", "name", "brand",
            "price", "in_stock", "valid_from", "valid_to", "is_current"]
for c in colonnes:
    if c not in h.columns:
        h[c] = None
donnees = h[colonnes].astype(object).where(pd.notnull(h[colonnes]), None)  # NaT/NaN -> None
lignes = list(donnees.itertuples(index=False, name=None))
execute_values(cur, f"INSERT INTO produit_historise ({','.join(colonnes)}) VALUES %s", lignes)
conn.commit()
print(len(lignes), "versions insérées dans produit_historise")

331056 versions insérées dans produit_historise


In [13]:
# Stat : combien de versions perd-on si on dédoublonne sur date_releve au lieu de scraped_at ?
n_scraped = df.drop_duplicates(["site", "cle", "scraped_at"]).shape[0]
n_date = df.drop_duplicates(["site", "cle", "date_releve"]).shape[0]
print(f"dédup scraped_at  : {n_scraped} versions  (choix retenu)")
print(f"dédup date_releve : {n_date} versions")
print(f"-> date_releve en garderait {n_scraped - n_date} de moins, "
      f"soit -{100 * (n_scraped - n_date) / n_scraped:.1f}%")

dédup scraped_at  : 331056 versions  (choix retenu)
dédup date_releve : 325677 versions
-> date_releve en garderait 5379 de moins, soit -1.6%


## Questions d'entraînement 

- Répondre aux questions ci-dessous avec SQL et pandas (i.e. deux implémentations pour chaque)
- Vous n'avez besoin que de la table `produit_historise` et ce notebook.

In [14]:
import warnings
warnings.filterwarnings("ignore", message=".*SQLAlchemy.*")
ph = pd.read_sql("SELECT * FROM produit_historise", conn)

ph["price"] = pd.to_numeric(ph["price"])  # conversion numeric SQL vers float pandas
print(ph.shape)

(331056, 11)


### Q1. (exemple) Combien de produits au catalogue aujourd'hui, par site ?

In [15]:
# --- SQL ---
display(pd.read_sql("""
    SELECT site, COUNT(*) AS n
    FROM produit_historise
    WHERE is_current
    GROUP BY site
    ORDER BY n DESC
""", conn))

# --- pandas ---
display(ph[ph.is_current].groupby("site").size()
          .sort_values(ascending=False).rename("n").reset_index())

,site,n
0,zooplus_fr,74975
1,animalis,22463
2,bitiba_fr,13634
3,clubvetshop,8564
4,maxizoo,8483
5,vetostore,5798
6,vetoplus,2810
7,chronovet,2488
8,pharmacy4pets,1378
9,univers_veto,354


,site,n
0,zooplus_fr,74975
1,animalis,22463
2,bitiba_fr,13634
3,clubvetshop,8564
4,maxizoo,8483
5,vetostore,5798
6,vetoplus,2810
7,chronovet,2488
8,pharmacy4pets,1378
9,univers_veto,354


### Q2. Prix actuel minimum, maximum et moyen par site.

In [16]:
# --- SQL ---
display(pd.read_sql("""
    SELECT site,
           MIN(price) AS mini,
           MAX(price) AS maxi,
           ROUND(AVG(price), 2) AS moyen
    FROM produit_historise
    WHERE is_current
    GROUP BY site
    ORDER BY site
""", conn))

# --- pandas ---
display(ph[ph.is_current].groupby("site")["price"].agg(["min", "max", "mean"]).round(2))

,site,mini,maxi,moyen
0,animalis,0.32,10799.00,78.40
1,bitiba_fr,0.29,882.99,34.03
2,chronovet,1.80,703.20,38.44
3,clubvetshop,0.16,845.52,31.61
4,maxizoo,NaN,NaN,NaN
5,pharmacy4pets,0.00,371.33,22.05
6,univers_veto,0.38,329.48,39.27
7,vetoplus,0.00,2463.60,35.80
8,vetostore,0.00,717.59,30.92
9,zooplus_fr,0.00,999.00,48.72


,min,max,mean
site,,,
animalis,0.32,10799.00,78.40
bitiba_fr,0.29,882.99,34.03
chronovet,1.80,703.20,38.44
clubvetshop,0.16,845.52,31.61
maxizoo,NaN,NaN,NaN
pharmacy4pets,0.00,371.33,22.05
univers_veto,0.38,329.48,39.27
vetoplus,0.00,2463.60,35.80
vetostore,0.00,717.59,30.92


### Q3. Combien de produits en rupture (`in_stock = false`) actuellement, par site ?

On peut utiliser `FILTER` avec un `WHERE` pour avoir un oneliner (i.e. un morceau de code qui tient sur une ligne).

In [17]:
# --- SQL ---
display(pd.read_sql("""
    SELECT site,
           COUNT(*) FILTER (WHERE in_stock IS FALSE) AS ruptures,
           COUNT(*) AS total
    FROM produit_historise
    WHERE is_current
    GROUP BY site
    ORDER BY site
""", conn))

# --- pandas ---
maintenant = ph[ph.is_current]
display(maintenant.assign(rupture=maintenant.in_stock == False)
                  .groupby("site").agg(ruptures=("rupture", "sum"), total=("rupture", "size")))

,site,ruptures,total
0,animalis,10043,22463
1,bitiba_fr,539,13634
2,chronovet,5,2488
3,clubvetshop,18,8564
4,maxizoo,0,8483
5,pharmacy4pets,174,1378
6,univers_veto,13,354
7,vetoplus,2196,2810
8,vetostore,813,5798
9,zooplus_fr,15712,74975


,ruptures,total
site,,
animalis,10043,22463
bitiba_fr,539,13634
chronovet,5,2488
clubvetshop,18,8564
maxizoo,0,8483
pharmacy4pets,174,1378
univers_veto,13,354
vetoplus,2196,2810
vetostore,813,5798


### Q4. L'historique complet d'un produit : toutes ses versions triées.

In [18]:
PRODUIT = ("vetoplus", "3552791071358")

# --- SQL ---
display(pd.read_sql("""
    SELECT valid_from, valid_to, price, is_current
    FROM produit_historise
    WHERE site = %(s)s AND cle = %(c)s
    ORDER BY valid_from
""", conn, params={"s": PRODUIT[0], "c": PRODUIT[1]}))

# --- pandas ---
display(ph[(ph.site == PRODUIT[0]) & (ph.cle == PRODUIT[1])]
          .sort_values("valid_from")[["valid_from", "valid_to", "price", "is_current"]])

,valid_from,valid_to,price,is_current
0,2026-07-12 14:59:19.001665+00:00,2026-07-18 16:16:15.119687+00:00,23.99,False
1,2026-07-18 16:16:15.119687+00:00,NaT,19.99,True


,valid_from,valid_to,price,is_current
123270,2026-07-12 14:59:19.001665+00:00,2026-07-18 16:16:15.119687+00:00,23.99,False
123271,2026-07-18 16:16:15.119687+00:00,NaT,19.99,True


### Q5. Quels produits ont le plus bougé (le plus de versions Par site) ?

In [19]:
# --- SQL ---
display(pd.read_sql("""
    SELECT site, cle, COUNT(*) AS versions
    FROM produit_historise
    GROUP BY site, cle
    ORDER BY versions DESC
    LIMIT 10
""", conn))

# --- pandas ---
display(ph.groupby(["site", "cle"]).size()
          .sort_values(ascending=False).head(10).rename("versions").reset_index())

,site,cle,versions
0,vetoplus,3760252080157,12
1,vetoplus,3661716109165,9
2,vetoplus,3661716103064,9
3,univers_veto,3283021955666,7
4,univers_veto,2000023886757,7
5,univers_veto,2000090091900,7
6,univers_veto,0086621771253,7
7,univers_veto,0086621465121,7
8,univers_veto,0086621771055,7
9,univers_veto,0086621465046,7


,site,cle,versions
0,vetoplus,3760252080157,12
1,vetoplus,3661716103064,9
2,vetoplus,3661716109165,9
3,univers_veto,3700454507328,7
4,univers_veto,3700454506413,7
5,univers_veto,3401177502187,7
6,univers_veto,3605874350861,7
7,univers_veto,3700454505744,7
8,univers_veto,3595890212901,7
9,univers_veto,3662952002204,7


In [20]:
# Variante Q5 : exactement UN produit par site, celui qui a le plus de versions.

# --- SQL ---
# DISTINCT ON (site) garde la première ligne de chaque site selon le ORDER BY,
# qui doit donc commencer par site, puis trier par versions décroissantes.
display(pd.read_sql("""
    SELECT DISTINCT ON (site) site, cle, COUNT(*) AS versions
    FROM produit_historise
    GROUP BY site, cle
    ORDER BY site, versions DESC, cle
""", conn))

# --- pandas ---
versions = ph.groupby(["site", "cle"]).size().rename("versions").reset_index()
display(versions.sort_values(["site", "versions", "cle"], ascending=[True, False, True])
                .groupby("site").head(1).reset_index(drop=True))

,site,cle,versions
0,animalis,0015561109987,5
1,bitiba_fr,0000042017097,3
2,chronovet,0000004503459,4
3,clubvetshop,0035585211176,4
4,maxizoo,2050000023620,4
5,pharmacy4pets,https://www.pharmacy4pets.fr/adtab-chat,4
6,univers_veto,0086621465046,7
7,vetoplus,3760252080157,12
8,vetostore,https://www.vetostore.com/eukanuba-adult-weigh...,4
9,zooplus_fr,0000000149426,3


,site,cle,versions
0,animalis,0015561109987,5
1,bitiba_fr,0000042017097,3
2,chronovet,0000004503459,4
3,clubvetshop,0035585211176,4
4,maxizoo,2050000023620,4
5,pharmacy4pets,https://www.pharmacy4pets.fr/adtab-chat,4
6,univers_veto,0086621465046,7
7,vetoplus,3760252080157,12
8,vetostore,https://www.vetostore.com/eukanuba-adult-weigh...,4
9,zooplus_fr,0000000149426,3


### Q6. Quel était le prix de tel produit à une date donnée `D` ?

In [21]:
D = "2026-07-15"

# --- SQL ---
display(pd.read_sql("""
    SELECT price
    FROM produit_historise
    WHERE site = %(s)s AND cle = %(c)s
      AND valid_from <= %(d)s
      AND (valid_to IS NULL OR valid_to > %(d)s)
""", conn, params={"s": "vetoplus", "c": "3552791071358", "d": D}))

# --- pandas ---
d = pd.Timestamp(D, tz="UTC")
m = ph[(ph.site == "vetoplus") & (ph.cle == "3552791071358")]
display(m[(m.valid_from <= d) & (m.valid_to.isna() | (m.valid_to > d))]["price"])

,price
0,23.99


123270    23.99
Name: price, dtype: float64

### Q7. Combien de produits au catalogue (catalogue composé de tous les produits de tous les sites, pas besoin de `groupby`) à une date passée `D` ?

- Utiliser `valid_from` et `valid_to` dans une clause `WHERE`, ainsi que la date `D`.

In [22]:
D = "2026-07-12"

# --- SQL ---
display(pd.read_sql("""
    SELECT COUNT(*) AS catalogue
    FROM produit_historise
    WHERE valid_from <= %(d)s
      AND (valid_to IS NULL OR valid_to > %(d)s)
""", conn, params={"d": D}))

# --- pandas ---
d = pd.Timestamp(D, tz="UTC")
present = (ph.valid_from <= d) & (ph.valid_to.isna() | (ph.valid_to > d))
print("catalogue au", D, ":", int(present.sum()))

,catalogue
0,86503


catalogue au 2026-07-12 : 86503


### Q8. (DIFFICILE) Top 10 des plus fortes variations de prix entre la première et la dernière version d'un produit pour un même site (euros et %).

In [23]:
# --- SQL ---
display(pd.read_sql("""
    WITH bornes AS (
        SELECT site, cle,
               FIRST_VALUE(price) OVER w AS prix_debut,
               LAST_VALUE(price)  OVER w AS prix_fin
        FROM produit_historise
        WINDOW w AS (
            PARTITION BY site, cle
            ORDER BY valid_from
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        )
    )
    SELECT DISTINCT site, cle, prix_debut, prix_fin,
           ROUND(prix_fin - prix_debut, 2) AS delta,
           ROUND(100 * (prix_fin - prix_debut) / NULLIF(prix_debut, 0), 1) AS delta_pct
    FROM bornes
    WHERE prix_debut <> prix_fin
    ORDER BY delta_pct DESC
    LIMIT 10
""", conn))

# --- pandas ---
g = ph.sort_values("valid_from").groupby(["site", "cle"])["price"]

# On calcule une dataframe avec les "bornes" (i.e. la première et la dernière)
bornes = pd.DataFrame(
    {
        "prix_debut": g.first(), 
        "prix_fin": g.last()
    }
)
bornes = bornes[bornes.prix_debut != bornes.prix_fin].copy()
bornes["delta"] = (bornes.prix_fin - bornes.prix_debut).round(2)
bornes["delta_pct"] = (100 * (bornes.prix_fin - bornes.prix_debut) / bornes.prix_debut).round(1)
display(bornes.sort_values("delta_pct", ascending=False).head(10))

,site,cle,prix_debut,prix_fin,delta,delta_pct
0,zooplus_fr,8683769632266,12.99,999.00,986.01,7590.5
1,zooplus_fr,8683769632273,13.99,999.00,985.01,7040.8
2,zooplus_fr,8683769632280,13.99,999.00,985.01,7040.8
3,zooplus_fr,8683769599583,15.55,999.00,983.45,6324.4
4,zooplus_fr,8032766455239,17.95,999.00,981.05,5465.5
5,zooplus_fr,8032766860224,19.95,999.00,979.05,4907.5
6,zooplus_fr,8421763353639,23.59,999.00,975.41,4134.8
7,zooplus_fr,8683769977626,25.59,999.00,973.41,3803.9
8,zooplus_fr,8032766964939,27.99,999.00,971.01,3469.1
9,zooplus_fr,8714265071362,2.79,71.49,68.70,2462.4


prix_debut  prix_fin   delta  delta_pct
site       cle                                                   
zooplus_fr 8683769632266       12.99    999.00  986.01     7590.5
           8683769632280       13.99    999.00  985.01     7040.8
           8683769632273       13.99    999.00  985.01     7040.8
           8683769599583       15.55    999.00  983.45     6324.4
           8032766455239       17.95    999.00  981.05     5465.5
           8032766860224       19.95    999.00  979.05     4907.5
           8421763353639       23.59    999.00  975.41     4134.8
           8683769977626       25.59    999.00  973.41     3803.9
           8032766964939       27.99    999.00  971.01     3469.1
           8714265071362        2.79     71.49   68.70     2462.4

In [29]:
# Autre solution pour Q8, bcp plus simple : pas de fonction WINDOW
# 💫 Idée : la dernière version, c'est déjà is_current. Donc on n'a qu'à trouver la première !

# --- SQL ---
# DISTINCT ON (site, cle) trié par valid_from donne la PREMIÈRE version de chaque produit.
# On la joint à sa version courante, qui est la dernière.
display(pd.read_sql("""
    WITH premiere AS (
        SELECT DISTINCT ON (site, cle) site, cle, price AS prix_debut
        FROM produit_historise
        ORDER BY site, cle, valid_from
    )
    SELECT p.site, p.cle, p.prix_debut, c.price AS prix_fin,
           ROUND(c.price - p.prix_debut, 2) AS delta,
           ROUND(100 * (c.price - p.prix_debut) / p.prix_debut, 1) AS delta_pct
    FROM premiere p
    JOIN produit_historise c
      ON c.site = p.site AND c.cle = p.cle AND c.is_current
    WHERE p.prix_debut <> c.price AND p.prix_debut <> 0
    ORDER BY delta_pct DESC
    LIMIT 10
""", conn))

,site,cle,prix_debut,prix_fin,delta,delta_pct
0,zooplus_fr,8683769632266,12.99,999.00,986.01,7590.5
1,zooplus_fr,8683769632273,13.99,999.00,985.01,7040.8
2,zooplus_fr,8683769632280,13.99,999.00,985.01,7040.8
3,zooplus_fr,8683769599583,15.55,999.00,983.45,6324.4
4,zooplus_fr,8032766455239,17.95,999.00,981.05,5465.5
5,zooplus_fr,8032766860224,19.95,999.00,979.05,4907.5
6,zooplus_fr,8421763353639,23.59,999.00,975.41,4134.8
7,zooplus_fr,8683769977626,25.59,999.00,973.41,3803.9
8,zooplus_fr,8032766964939,27.99,999.00,971.01,3469.1
9,zooplus_fr,8714265071362,2.79,71.49,68.70,2462.4


Quelques explications supplémentaires : 

Explications de :
```sql
JOIN produit_historise c ON c.site = p.site AND c.cle = p.cle AND c.is_current
```
- `p` = première version du produit :1 ligne par produit, via la CTE premiere (i.e. la sous requête `premiere`)
- `c.site = p.site AND c.cle = p.cle` : on raccorde c au même produit
- `c.is_current` : booléen, équivaut à = true : ne garde que la version courante de `c`, donc la dernière.
- `is_current` est vrai sur une seule ligne par produit : la jointure donne donc 1 ligne par produit, `prix_debut` (de `p`) face au prix courant `c.price`
- Sans `is_current`, `p` s'apparierait à toutes les versions de `c`  (donc si, 5 versions, on obtiendrait 5 lignes)

Explications de : 

> `is_current` dans le `ON`, pas dans le `WHERE`

- **En jointure interne (i.e. sur la même table), `ON` et `WHERE` donnent ici le même résultat.** C'est donc juste un choix de le mettre dans le `ON` 
- `is_current` dit quelle ligne de `c` joindre. Ce n'est pas un filtre du résultat. Donc on préfère le mettre dans le `ON` (même si ça ne change rien en comparaison avec le cas dans le `WHERE`)
- **La distinction compterait pour un `LEFT JOIN`**. 


Explications de : 
```sql
WHERE p.prix_debut <> c.price AND p.prix_debut <> 0
```
- `p.prix_debut <> c.price` : garde les produits dont le prix a changé + écarte les prix inchangés (i.e. le delta prix est  nul)
- `p.prix_debut <> 0` : évite la division par zéro dans `delta_pct` (.../ `p.prix_debut`), comme `NULLIF(prix_debut, 0)` de la version fenêtre.

Remarque : les `NULL` sont bien gérés

- Une comparaison avec `NULL` ne vaut jamais `TRUE` (elle vaut `NULL`), et le `WHERE` ne garde que le `TRUE`.
- Donc un prix à `NULL` fait échouer la comparaison avec `<>` et exclut la ligne.

### Q9. Pour chaque produit, la variation de prix d'une version à la suivante (`LAG`).

Quand le prix ne bouge pas entre deux relevés (i.e. deux observations), la variation vaut 0.

In [26]:
# --- SQL ---
display(pd.read_sql("""
    SELECT site, cle, valid_from, price,
           price - LAG(price) OVER (PARTITION BY site, cle ORDER BY valid_from) AS variation
    FROM produit_historise
    WHERE site = 'vetoplus'
    ORDER BY cle, valid_from
    LIMIT 100
""", conn))

# --- pandas ---
v = ph[ph.site == "vetoplus"].sort_values(["cle", "valid_from"]).copy()
v["variation"] = v.groupby("cle")["price"].diff()
display(v[["cle", "valid_from", "price", "variation"]].head(100))

,site,cle,valid_from,price,variation
0,vetoplus,0000001024296,2026-04-01 20:21:00.950727+00:00,21.9,NaN
1,vetoplus,0000001024296,2026-07-12 13:12:21.492880+00:00,21.9,0.0
2,vetoplus,0000001024296,2026-07-18 14:25:59.731569+00:00,21.9,0.0
3,vetoplus,0012575198464,2026-07-12 14:54:58.604293+00:00,12.9,NaN
4,vetoplus,0012575198464,2026-07-18 16:11:54.150585+00:00,12.9,0.0
...,...,...,...,...,...
95,vetoplus,0086621465046,2026-07-12 13:12:25.667239+00:00,169.9,11.5
96,vetoplus,0086621465046,2026-07-18 14:26:04.588747+00:00,169.9,0.0
97,vetoplus,0086621465121,2026-04-01 20:20:52.761154+00:00,63.2,NaN
98,vetoplus,0086621465121,2026-07-12 13:12:27.960034+00:00,62.9,-0.3


,cle,valid_from,price,variation
121722,0000001024296,2026-04-01 20:21:00.950727+00:00,21.9,NaN
121723,0000001024296,2026-07-12 13:12:21.492880+00:00,21.9,0.0
121724,0000001024296,2026-07-18 14:25:59.731569+00:00,21.9,0.0
121725,0012575198464,2026-07-12 14:54:58.604293+00:00,12.9,NaN
121726,0012575198464,2026-07-18 16:11:54.150585+00:00,12.9,0.0
...,...,...,...,...
121815,0086621465046,2026-07-12 13:12:25.667239+00:00,169.9,11.5
121816,0086621465046,2026-07-18 14:26:04.588747+00:00,169.9,0.0
121817,0086621465121,2026-04-01 20:20:52.761154+00:00,63.2,NaN
121818,0086621465121,2026-07-12 13:12:27.960034+00:00,62.9,-0.3


### Q10. (DIFFICILE) Matching inter-sites : pour un même `ean`, quel écart de prix entre sites aujourd'hui ?

- Attention, il sera difficile d'interpéter toutes les valeurs à ce stade, en particulier quand le même `ean` est utilisé pour un produit et des lots de ce produit.

In [31]:
# --- SQL ---
# C'est cette ligne à laquelle il faut bien penser : 
# AND ean IS NOT NULL AND ean <> 
display(pd.read_sql("""
    SELECT ean,
           COUNT(DISTINCT site) AS n_sites,
           MIN(price) AS moins_cher,
           MAX(price) AS plus_cher,
           ROUND(MAX(price) - MIN(price), 2) AS ecart
    FROM produit_historise
    WHERE is_current AND ean IS NOT NULL AND ean <> ''
    GROUP BY ean
    HAVING COUNT(DISTINCT site) > 1
    ORDER BY ecart DESC
    LIMIT 10
""", conn))

# --- pandas ---
# C'est ce mask qui est considérée par facile : 
# ph.ean.notna() & (ph.ean.astype(str).str.strip() != ""
maintenant = ph[ph.is_current & ph.ean.notna() & (ph.ean.astype(str).str.strip() != "")]
inter = maintenant.groupby("ean").agg(n_sites=("site", "nunique"),
                                      moins_cher=("price", "min"), plus_cher=("price", "max"))
inter = inter[inter.n_sites > 1].copy()
inter["ecart"] = (inter.plus_cher - inter.moins_cher).round(2)
display(inter.sort_values("ecart", ascending=False).head(10))

,ean,n_sites,moins_cher,plus_cher,ecart
0,4011905127965,2,245.47,998.71,753.24
1,5414365257439,2,199.99,529.66,329.67
2,4027718011476,2,39.49,324.99,285.50
3,4260704280917,2,94.99,363.99,269.00
4,6975069305523,2,635.59,899.99,264.40
5,3515650002832,2,452.00,703.20,251.20
6,4260704280788,2,85.99,329.99,244.00
7,4260704280764,2,85.99,329.99,244.00
8,4260704280924,2,76.49,293.99,217.50
9,8414042003882,2,231.99,447.99,216.00


,n_sites,moins_cher,plus_cher,ecart
ean,,,,
4011905127965,2,245.47,998.71,753.24
5414365257439,2,199.99,529.66,329.67
4027718011476,2,39.49,324.99,285.50
4260704280917,2,94.99,363.99,269.00
6975069305523,2,635.59,899.99,264.40
3515650002832,2,452.00,703.20,251.20
4260704280788,2,85.99,329.99,244.00
4260704280764,2,85.99,329.99,244.00
4260704280924,2,76.49,293.99,217.50
